# 3-Site Glioma IDH Cohort — Data Split Definition

**This notebook is the single, citable source of truth for how the train/val/test splits used in all
experiments were generated.** Re-running it top-to-bottom regenerates the exact CSVs in
`data/preprocessed/v1.0.0/_global/splits/` (deterministic given `seed=42` and the same input metadata).

- **Author artifact for the paper**: cite this notebook (and the frozen CSVs) in the Methods section.
- **Inputs** (metadata only — no image processing):
  - `data/metadata/UCSF-PDGM-metadata_v5.csv`
  - `data/metadata/UTSW_Glioma_Metadata.tsv`
  - `data/metadata/UPENN-GBM_clinical_info_v2.1.csv`, `UPENN-GBM_acquisition.csv`
  - `data/preprocessed/v1.0.0/utsw-glioma/_global/seg_source_inventory.csv` (UTSW QC gate)
- **Outputs**: `splits_loso_foldA.csv`, `splits_loso_foldB.csv`, `splits_vendor_philips.csv`,
  `splits_field.csv`, `splits_random_5fold.csv` (+ this notebook as documentation).

## 1. Design rationale

The scientific target is **IDH mutation status** prediction from MRI. The split design is built to
measure **out-of-distribution generalization across sites**, because that is the deployment reality.

**Primary evaluation — site leave-one-site-out (LOSO), 2 folds.**
- Fold A: test = **UCSF**, train = UTSW + UPenn.
- Fold B: test = **UTSW**, train = UCSF + UPenn.
- **UPenn is never a test site.** It has only 16 IDH-Mut cases, so using it as a held-out site would
  make positive-class metrics (AUC/recall on Mut) extremely unstable. It always contributes to training.

**Auxiliary evaluation 1 — leave-one-vendor-out inside UTSW.**
UTSW is the only multi-vendor site. Train on Siemens+GE (minority vendors Hitachi/Toshiba/Not-Reported
also go to train), test on **Philips**. This isolates *scanner-manufacturer* domain shift from
*site/population* shift.

**Auxiliary evaluation 2 — field-strength shift inside UTSW.**
Test on **3T**, train on the rest (1.5T + minority/Not-Reported field strengths). Isolates
*field-strength* domain shift.

**Baseline — random stratified 5-fold (all sites mixed).**
The optimistic in-distribution reference. The gap between this and LOSO quantifies the domain-shift penalty.

**Invariants enforced in every split:**
1. **Patient-level grouping** — no patient appears in both train and test. UCSF has 6 patients with a
   baseline + follow-up session; both sessions are always kept on the same side.
2. **IDH stratification** — Mut/WT ratio is preserved across roles where possible.
3. **Site ratio** in the random split — stratification key is `site × IDH`, so both site mix and class
   balance are held per fold.
4. **IDH-NA fully excluded** — UTSW 3 NA and UPenn 96 `NOS/NEC` (indeterminate) are dropped before splitting.

**Inner validation:** each outer `train` is further split 80/20 into inner-train / `val`
(patient-grouped, `site×IDH`-stratified). Hyperparameters and early stopping use `val`; the outer
`test` is touched exactly once. **`seed = 42`** everywhere.

In [1]:
import pandas as pd, numpy as np, re, os
from pathlib import Path

from sklearn.model_selection import StratifiedGroupKFold

SEED = 42

# Resolve repo root from the notebook location so this is portable if the repo moves.
root = Path.cwd()
for _ in range(8):
    if (root / "data" / "metadata").exists():
        break
    root = root.parent
assert (root / "data" / "metadata").exists(), "could not locate repo root containing data/metadata"

MD   = root / "data" / "metadata"
PROC = root / "data" / "preprocessed" / "v1.0.0"
OUT  = PROC / "_global" / "splits"
OUT.mkdir(parents=True, exist_ok=True)
print("repo root :", root)
print("outputs -> :", OUT)

repo root : /home/llmteam0203/Scripts/python/OpenIDH
outputs -> : /home/llmteam0203/Scripts/python/OpenIDH/data/preprocessed/v1.0.0/_global/splits


## 2. Load cohorts and apply per-site IDH rules

Each site encodes IDH differently, so mapping is explicit per source:

| site | IDH column | → WT | → Mut | → dropped (NA) |
|------|-----------|------|-------|----------------|
| UCSF | `IDH` | `wildtype` | any molecular call (`IDH1 p.R132H`, `mutated (NOS)`, …) | none present |
| UTSW | `IDH` | `wild type` | `mutated` | 3 blank/NA |
| UPenn| `IDH1`| `Wildtype` | `Mutated` | 96 `NOS/NEC` (indeterminate) |

Note `mutated (NOS)` at **UCSF** means *mutation confirmed, subtype unspecified* → **Mut**, whereas
`NOS/NEC` at **UPenn** means *IDH could not be classified* → **NA**. Opposite meaning; handled separately.

In [2]:
def _num(x):
    m = re.search(r"(\d+)", str(x));  return int(m.group(1)) if m else None

# ---- UCSF: all 501 metadata sessions; patient key = numeric ID ----
u = pd.read_csv(MD / "UCSF-PDGM-metadata_v5.csv")
def ucsf_idh(x):
    s = str(x).strip().lower()
    return "WT" if s in ("wildtype", "wild type", "wt") else "Mut"  # remainder are genuine mutations
ucsf = pd.DataFrame({
    "subject_id": u["ID"].astype(str),
    "site": "UCSF",
    "idh":  u["IDH"].map(ucsf_idh),
    "patient_id": "UCSF-" + u["ID"].map(_num).astype(str),
    "vendor": "NA", "field": "NA",
})

# ---- UTSW: qc_passed==true (622) then drop IDH-NA -> 619 ----
inv = pd.read_csv(PROC / "utsw-glioma" / "_global" / "seg_source_inventory.csv")
inv = inv[inv["qc_passed"].astype(str).str.lower().isin(["true", "1"])]
qc_ids = set(inv["subject_id"])
t = pd.read_csv(MD / "UTSW_Glioma_Metadata.tsv", sep="\t")
t = t[t["Subject ID"].isin(qc_ids)].copy()
def utsw_idh(x):
    s = str(x).strip().lower()
    if "wild" in s: return "WT"
    if "mut"  in s: return "Mut"
    return "NA"
t["idh"] = t["IDH"].map(utsw_idh)
t = t[t["idh"] != "NA"].copy()
utsw = pd.DataFrame({
    "subject_id": t["Subject ID"].astype(str),
    "site": "UTSW",
    "idh":  t["idh"].values,
    "patient_id": "UTSW-" + t["Subject ID"].astype(str),
    "vendor": t["Scanner Make"].astype(str).str.strip().values,
    "field":  t["Scanner Strength"].astype(str).str.strip().values,
})

# ---- UPenn: clinical ∩ preprocessed images (611) then drop NOS/NEC -> 515 ----
cl  = pd.read_csv(MD / "UPENN-GBM_clinical_info_v2.1.csv")
acq = pd.read_csv(MD / "UPENN-GBM_acquisition.csv")[["ID", "Manufacturer", "Magnetic Field Strength"]]
imaged = {d for d in os.listdir(PROC / "upenn-gbm")
          if (PROC / "upenn-gbm" / d).is_dir() and d.startswith("UPENN")}
cl = cl[cl["ID"].isin(imaged)].copy()
def upenn_idh(x):
    s = str(x).strip().lower()
    if s == "wildtype": return "WT"
    if s == "mutated":  return "Mut"
    return "NA"          # NOS/NEC etc.
cl["idh"] = cl["IDH1"].map(upenn_idh)
cl = cl[cl["idh"] != "NA"].merge(acq, on="ID", how="left")
upenn = pd.DataFrame({
    "subject_id": cl["ID"].astype(str),
    "site": "UPenn",
    "idh":  cl["idh"].values,
    "patient_id": "UPenn-" + cl["ID"].astype(str).str.replace(r"_\d+$", "", regex=True),
    "vendor": cl["Manufacturer"].astype(str).str.strip().values,
    "field":  cl["Magnetic Field Strength"].astype(str).str.strip().values,
})

df = pd.concat([ucsf, utsw, upenn], ignore_index=True)
df["y"]     = (df["idh"] == "Mut").astype(int)
df["strat"] = df["site"] + "_" + df["idh"]
len(df)

1635

## 3. Cohort sanity check & patient grouping

Confirm the frozen cohort numbers and that patient-level grouping is correct (only UCSF has
multi-session patients).

In [3]:
def cc(d): return dict(n=len(d), Mut=int((d.idh=="Mut").sum()), WT=int((d.idh=="WT").sum()))
print("cohort counts:")
for s in ["UCSF","UTSW","UPenn"]:
    print(f"  {s:6s}", cc(df[df.site==s]))
print("  TOTAL ", cc(df), f"pos_rate={df.y.mean():.3f}")

print("\npatient grouping:")
for s in ["UCSF","UTSW","UPenn"]:
    d = df[df.site==s]; vc = d.patient_id.value_counts(); multi = vc[vc>1]
    print(f"  {s:6s} patients={d.patient_id.nunique()} sessions={len(d)} multi-session={multi.to_dict()}")

assert cc(df) == dict(n=1635, Mut=295, WT=1340), "cohort counts drifted from the frozen definition!"

cohort counts:
  UCSF   {'n': 501, 'Mut': 103, 'WT': 398}
  UTSW   {'n': 619, 'Mut': 176, 'WT': 443}
  UPenn  {'n': 515, 'Mut': 16, 'WT': 499}
  TOTAL  {'n': 1635, 'Mut': 295, 'WT': 1340} pos_rate=0.180

patient grouping:
  UCSF   patients=495 sessions=501 multi-session={'UCSF-429': 2, 'UCSF-396': 2, 'UCSF-409': 2, 'UCSF-431': 2, 'UCSF-391': 2, 'UCSF-433': 2}
  UTSW   patients=619 sessions=619 multi-session={}
  UPenn  patients=515 sessions=515 multi-session={}


## 4. Split helpers — inner 80/20 (patient-grouped, stratified)

`StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)` → first fold (~20%) becomes `val`.
This guarantees no patient crosses the inner-train/val boundary and that `site×IDH` balance is kept.

In [4]:
def inner_split(train_df, seed=SEED):
    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    for tr, va in sgkf.split(train_df, train_df["strat"], train_df["patient_id"]):
        return train_df.index[tr], train_df.index[va]   # first fold -> ~20% val

def assign_roles(train_df, test_df, seed=SEED):
    roles = pd.Series("train", index=pd.concat([train_df, test_df]).index)
    _, iva = inner_split(train_df, seed)
    roles.loc[iva] = "val"
    roles.loc[test_df.index] = "test"
    return roles

def write_split(roles, fold_id, fname):
    out = df.loc[roles.index, ["subject_id","site","idh"]].copy()
    out["split_role"] = roles.values
    out["fold_id"]    = fold_id
    out.to_csv(OUT / fname, index=False)
    return out

def summarize(roles, label):
    rows=[]
    for r in ["train","val","test"]:
        sub = df.loc[roles[roles==r].index]
        if len(sub)==0: continue
        mut=int((sub.idh=="Mut").sum()); wt=int((sub.idh=="WT").sum())
        rows.append(dict(split=label, role=r, n=len(sub), Mut=mut, WT=wt,
                         pos=f"{mut/len(sub)*100:.1f}%"))
    return rows

summary = []

## 5. Primary — site LOSO (Fold A: test=UCSF, Fold B: test=UTSW)
UPenn stays in `train` in both folds.

In [5]:
for fold, test_site in [("A","UCSF"), ("B","UTSW")]:
    test_df  = df[df.site == test_site]
    train_df = df[df.site != test_site]
    roles = assign_roles(train_df, test_df)
    write_split(roles, f"loso_{fold}", f"splits_loso_fold{fold}.csv")
    summary += summarize(roles, f"LOSO Fold {fold} (test={test_site})")
pd.DataFrame([r for r in summary if r["split"].startswith("LOSO")])

,split,role,n,Mut,WT,pos
0,LOSO Fold A (test=UCSF),train,907,154,753,17.0%
1,LOSO Fold A (test=UCSF),val,227,38,189,16.7%
2,LOSO Fold A (test=UCSF),test,501,103,398,20.6%
3,LOSO Fold B (test=UTSW),train,812,95,717,11.7%
4,LOSO Fold B (test=UTSW),val,204,24,180,11.8%
5,LOSO Fold B (test=UTSW),test,619,176,443,28.4%


## 6. Auxiliary 1 — UTSW leave-one-vendor-out (test = Philips)
Minority vendors (Hitachi/Toshiba/Not-Reported) go to the train side.

In [6]:
utsw_df = df[df.site == "UTSW"]
print("UTSW vendor counts:", utsw_df.vendor.value_counts().to_dict())
philips = utsw_df[utsw_df.vendor.str.contains("Philips", case=False, na=False)]
train_v = utsw_df.drop(philips.index)
roles_v = assign_roles(train_v, philips)
write_split(roles_v, "vendor_philips", "splits_vendor_philips.csv")
summary += summarize(roles_v, "Vendor (UTSW: test=Philips)")
pd.DataFrame([r for r in summary if r["split"].startswith("Vendor")])

UTSW vendor counts: {'Siemens': 265, 'GE': 170, 'Philips': 140, 'Not Reported': 28, 'Hitachi': 15, 'Toshiba': 1}


,split,role,n,Mut,WT,pos
0,Vendor (UTSW: test=Philips),train,383,102,281,26.6%
1,Vendor (UTSW: test=Philips),val,96,25,71,26.0%
2,Vendor (UTSW: test=Philips),test,140,49,91,35.0%


## 7. Auxiliary 2 — UTSW field strength (test = 3T, train = rest)
Minority / Not-Reported field strengths go to the train side.

In [7]:
def is3T(x):
    try: return abs(float(x) - 3.0) < 1e-6
    except: return False
print("UTSW field counts:", utsw_df.field.value_counts().to_dict())
three   = utsw_df[utsw_df.field.map(is3T)]
train_f = utsw_df.drop(three.index)
roles_f = assign_roles(train_f, three)
write_split(roles_f, "field_3T", "splits_field.csv")
summary += summarize(roles_f, "Field (UTSW: test=3T, train=rest)")
pd.DataFrame([r for r in summary if r["split"].startswith("Field")])

UTSW field counts: {'1.5': 379, '3': 195, 'Not Reported': 28, '0.7': 6, '1.16': 5, '1': 3, '0.3': 2, '0.94999999': 1}


,split,role,n,Mut,WT,pos
0,"Field (UTSW: test=3T, train=rest)",train,339,99,240,29.2%
1,"Field (UTSW: test=3T, train=rest)",val,85,24,61,28.2%
2,"Field (UTSW: test=3T, train=rest)",test,195,53,142,27.2%


## 8. Baseline — all-site random stratified group 5-fold
Stratified on `site×IDH`, grouped by patient. Each subject is `test` in exactly one fold.

In [8]:
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
rand_out = []
for k, (tr, te) in enumerate(sgkf.split(df, df["strat"], df["patient_id"])):
    roles = assign_roles(df.iloc[tr], df.iloc[te])
    o = df.loc[roles.index, ["subject_id","site","idh"]].copy()
    o["split_role"] = roles.values; o["fold_id"] = k
    rand_out.append(o)
    summary += summarize(roles, f"Random 5-fold [fold {k}]")
pd.concat(rand_out).to_csv(OUT / "splits_random_5fold.csv", index=False)
pd.DataFrame([r for r in summary if r["split"].startswith("Random")])

,split,role,n,Mut,WT,pos
0,Random 5-fold [fold 0],train,1046,188,858,18.0%
1,Random 5-fold [fold 0],val,262,48,214,18.3%
2,Random 5-fold [fold 0],test,327,59,268,18.0%
3,Random 5-fold [fold 1],train,1046,189,857,18.1%
4,Random 5-fold [fold 1],val,262,47,215,17.9%
5,Random 5-fold [fold 1],test,327,59,268,18.0%
6,Random 5-fold [fold 2],train,1046,189,857,18.1%
7,Random 5-fold [fold 2],val,262,47,215,17.9%
8,Random 5-fold [fold 2],test,327,59,268,18.0%
9,Random 5-fold [fold 3],train,1046,189,857,18.1%


## 9. Full count table (train = inner-train, val = inner-val, test)

In [9]:
pd.DataFrame(summary)[["split","role","n","Mut","WT","pos"]]

,split,role,n,Mut,WT,pos
0,LOSO Fold A (test=UCSF),train,907,154,753,17.0%
1,LOSO Fold A (test=UCSF),val,227,38,189,16.7%
2,LOSO Fold A (test=UCSF),test,501,103,398,20.6%
3,LOSO Fold B (test=UTSW),train,812,95,717,11.7%
4,LOSO Fold B (test=UTSW),val,204,24,180,11.8%
5,LOSO Fold B (test=UTSW),test,619,176,443,28.4%
6,Vendor (UTSW: test=Philips),train,383,102,281,26.6%
7,Vendor (UTSW: test=Philips),val,96,25,71,26.0%
8,Vendor (UTSW: test=Philips),test,140,49,91,35.0%
9,"Field (UTSW: test=3T, train=rest)",train,339,99,240,29.2%


## 10. Leak / integrity checks (asserted — must all pass)
For every split & fold: (1) no `subject_id` shared between train/val and test,
(2) no patient split across the train/val vs test boundary, (3) no IDH-NA present.

In [10]:
files = {
 "loso_foldA":"splits_loso_foldA.csv", "loso_foldB":"splits_loso_foldB.csv",
 "vendor_philips":"splits_vendor_philips.csv", "field":"splits_field.csv",
 "random_5fold":"splits_random_5fold.csv",
}
sub2pat = dict(zip(df.subject_id, df.patient_id))
known   = set(df.subject_id)
all_ok  = True
for name, fname in files.items():
    s = pd.read_csv(OUT / fname)
    for fid in s.fold_id.unique():
        sf = s[s.fold_id == fid]
        tr = set(sf[sf.split_role=="train"].subject_id)
        va = set(sf[sf.split_role=="val"].subject_id)
        te = set(sf[sf.split_role=="test"].subject_id)
        overlap   = (tr & te) | (va & te) | (tr & va)
        pat_leak  = {sub2pat[i] for i in tr|va} & {sub2pat[i] for i in te}
        na        = [i for i in sf.subject_id if i not in known] + list(sf[sf.idh=="NA"].subject_id)
        ok = not overlap and not pat_leak and not na
        all_ok &= ok
        print(f"  {name:15s} fold={str(fid):14s} overlap={len(overlap)} patient_leak={len(pat_leak)} NA={len(na)} -> {'OK' if ok else 'FAIL'}")
assert all_ok, "integrity checks failed"
print("\nALL CHECKS PASSED — splits written to", OUT)

  loso_foldA      fold=loso_A         overlap=0 patient_leak=0 NA=0 -> OK
  loso_foldB      fold=loso_B         overlap=0 patient_leak=0 NA=0 -> OK
  vendor_philips  fold=vendor_philips overlap=0 patient_leak=0 NA=0 -> OK
  field           fold=field_3T       overlap=0 patient_leak=0 NA=0 -> OK
  random_5fold    fold=0              overlap=0 patient_leak=0 NA=0 -> OK
  random_5fold    fold=1              overlap=0 patient_leak=0 NA=0 -> OK
  random_5fold    fold=2              overlap=0 patient_leak=0 NA=0 -> OK
  random_5fold    fold=3              overlap=0 patient_leak=0 NA=0 -> OK
  random_5fold    fold=4              overlap=0 patient_leak=0 NA=0 -> OK

ALL CHECKS PASSED — splits written to /workspace/data/preprocessed/v1.0.0/_global/splits
